# arcroad — Street-level & terrain probe (v1.1)

**What this answers:** can arcroad show real street-level imagery at the places it says are severed,
and what does the globe view have available underneath it — using only data we are allowed to use.

**Hard constraint, restated.** Decision D-02: **zero Google data, ever.** No Street View, no scraping,
no tracing, no screenshots, no Maps tiles. Nothing in this notebook touches a Google property except
Open Buildings, which is separately licensed CC BY 4.0 / ODbL and is only checked for coverage here.

This notebook **probes and reports**. It downloads no imagery and publishes nothing.


## 1. Configuration

`MAPILLARY_TOKEN` is the only credential needed and it is free:
sign in at mapillary.com, then Developers → register an application → copy the client token.

Leave it empty and the Mapillary section will skip itself and say so. The other probes need nothing.

In [ ]:
MAPILLARY_TOKEN = ""   # paste a free Mapillary client token here, or leave empty to skip

# Real points from the v1.0 run, 25 Oct 2026. These are the places the method flags.
TARGETS = [
    ("DRIVE TARGET (Isnapur)",   17.5274, 78.2685, "21 m apart, model says 2,344 m"),
    ("Patancheru zone centre",   17.5300, 78.2800, "zone median DI 69.4"),
    ("Worst point, west",        17.4124, 78.3153, "12 m apart, model says 24,157 m"),
    ("Worst point, east",        17.5152, 78.6476, "20 m apart, model says 24,729 m"),
    ("Worst point, north",       17.5784, 78.5699, "20 m apart, model says 24,643 m"),
    ("Worst point, far south",   17.0754, 78.4898, "14 m apart, model says 14,416 m"),
    ("Core city reference",      17.3850, 78.4867, "core median DI 17.2"),
    ("Shamshabad sector",        17.2403, 78.4294, "worst ORR sector, median 55.4"),
]

RADIUS_M = 150          # how close an image must be to count as covering the point
HYD_BBOX = (78.050, 17.050, 78.850, 17.750)   # the measured area

import math, json, time
import urllib.request, urllib.error

def box(lat, lon, m):
    dlat = m / 110540.0
    dlon = m / (111320.0 * math.cos(math.radians(lat)))
    return (lon - dlon, lat - dlat, lon + dlon, lat + dlat)

def get(url, timeout=25):
    req = urllib.request.Request(url, headers={"User-Agent": "arcroad-probe/1.1 (civic research)"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.status, r.read()

print(f"{len(TARGETS)} points to probe")
print(f"Mapillary token: {'present' if MAPILLARY_TOKEN else 'NOT SET - that section will skip'}")

## 2. Mapillary — crowdsourced street-level imagery, CC BY-SA 4.0

The largest open street-level archive. Images are contributed by the public and licensed CC BY-SA,
which is compatible with publishing alongside ODbL data as long as attribution is carried.

We are asking one question per point: **is there any photograph of this place, and how old is it?**

In [ ]:
results_mapillary = {}

if not MAPILLARY_TOKEN:
    print("SKIPPED - no token. Everything below this cell still runs.")
else:
    for name, lat, lon, note in TARGETS:
        w, s, e, n = box(lat, lon, RADIUS_M)
        url = ("https://graph.mapillary.com/images"
               f"?access_token={MAPILLARY_TOKEN}"
               f"&fields=id,captured_at,is_pano,compass_angle"
               f"&bbox={w:.6f},{s:.6f},{e:.6f},{n:.6f}&limit=100")
        try:
            status, body = get(url)
            data = json.loads(body).get("data", [])
            years = sorted({int(d["captured_at"]) // 1000 // 31557600 + 1970
                            for d in data if d.get("captured_at")})
            panos = sum(1 for d in data if d.get("is_pano"))
            results_mapillary[name] = {"n": len(data), "panos": panos,
                                       "years": years[-3:] if years else []}
            print(f"{name:28s} {len(data):4d} images  {panos:3d} panoramic  years {years[-3:] if years else '-'}")
        except urllib.error.HTTPError as ex:
            print(f"{name:28s} HTTP {ex.code} - {ex.reason}")
            results_mapillary[name] = {"error": f"HTTP {ex.code}"}
        except Exception as ex:
            print(f"{name:28s} FAILED - {type(ex).__name__}: {ex}")
            results_mapillary[name] = {"error": str(ex)}
        time.sleep(0.4)

## 3. Panoramax — open street-level platform, no token required

A newer open-source alternative, run as a federation of instances. The central instance exposes a
STAC search API. If this has Indian coverage it is attractive, because it needs no commercial account.

In [ ]:
results_panoramax = {}
PANO_BASE = "https://api.panoramax.xyz/api/search"

for name, lat, lon, note in TARGETS[:4]:
    w, s, e, n = box(lat, lon, RADIUS_M)
    url = f"{PANO_BASE}?bbox={w:.6f},{s:.6f},{e:.6f},{n:.6f}&limit=50"
    try:
        status, body = get(url)
        feats = json.loads(body).get("features", [])
        results_panoramax[name] = {"n": len(feats)}
        print(f"{name:28s} {len(feats):4d} images (HTTP {status})")
    except urllib.error.HTTPError as ex:
        print(f"{name:28s} HTTP {ex.code} - endpoint may have moved; report the code, do not guess")
        results_panoramax[name] = {"error": f"HTTP {ex.code}"}
    except Exception as ex:
        print(f"{name:28s} FAILED - {type(ex).__name__}: {ex}")
        results_panoramax[name] = {"error": str(ex)}
    time.sleep(0.4)

print()
print("Whole-city sweep:")
w, s, e, n = HYD_BBOX
try:
    status, body = get(f"{PANO_BASE}?bbox={w},{s},{e},{n}&limit=1")
    print("  HTTP", status, "- returned", len(json.loads(body).get("features", [])), "of an unknown total")
except Exception as ex:
    print("  FAILED -", type(ex).__name__, ex)

## 4. OpenStreetMap's own view of what exists on the ground

Not imagery, but it tells us whether the open map already knows about pedestrian crossings,
footbridges and underpasses near our worst points. Where it does not, that is an OSM contribution
waiting to be made — and it is the cheapest way to improve every navigation app in the city,
not only ours.

Uses the Overpass API. Free, no token, rate-limited, so be gentle.

In [ ]:
OVERPASS = "https://overpass-api.de/api/interpreter"
import urllib.parse

def overpass(query, timeout=90):
    data = urllib.parse.urlencode({"data": query}).encode()
    req = urllib.request.Request(OVERPASS, data=data,
                                 headers={"User-Agent": "arcroad-probe/1.1 (civic research)"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read())

results_osm = {}
for name, lat, lon, note in TARGETS[:5]:
    q = f'''[out:json][timeout:60];
(
  node(around:400,{lat},{lon})["highway"="crossing"];
  way(around:400,{lat},{lon})["highway"="footway"]["footway"="crossing"];
  way(around:400,{lat},{lon})["bridge"]["highway"~"footway|path"];
  way(around:400,{lat},{lon})["tunnel"]["highway"~"footway|path"];
);
out count;'''
    try:
        res = overpass(q)
        cnt = int(res.get("elements", [{}])[0].get("tags", {}).get("total", 0))
        results_osm[name] = cnt
        verdict = "nothing recorded" if cnt == 0 else f"{cnt} crossing features"
        print(f"{name:28s} {verdict}")
    except Exception as ex:
        print(f"{name:28s} FAILED - {type(ex).__name__}: {ex}")
        results_osm[name] = None
    time.sleep(3)   # Overpass is a shared free service. Do not hammer it.

## 5. Terrain for the globe

The globe descent needs elevation only if we want relief. Copernicus DEM (30 m, free, open licence)
and SRTM are both usable. This checks that a terrain source responds for Hyderabad at all — it does
not download a tile set.

In [ ]:
# Open-Elevation: free, open, no token. Enough to confirm a usable elevation source exists.
try:
    pts = "|".join(f"{lat},{lon}" for _, lat, lon, _ in TARGETS[:5])
    status, body = get(f"https://api.open-elevation.com/api/v1/lookup?locations={pts}")
    for r in json.loads(body).get("results", []):
        print(f"  {r['latitude']:.4f}, {r['longitude']:.4f}  ->  {r['elevation']} m")
    print()
    print("Hyderabad sits on the Deccan plateau: expect 450-620 m and very little local relief.")
    print("If the spread here is under ~100 m, terrain adds nothing to the globe and should be")
    print("left out. Flat relief rendered dramatically is a lie told with a shader.")
except Exception as ex:
    print("FAILED -", type(ex).__name__, ex)

## 6. Google Open Buildings 2.5D — coverage check only

Licensed CC BY 4.0 or ODbL, which is why it is permissible where Street View is not. We are only
confirming Hyderabad is in the released tiles. **Remember D-03 as amended:** anything derived from
this is sub-metre lineage and must live in India-region storage, not on Cloudflare's global edge.
That is why building extrusion is v2 and not v1.

In [ ]:
# The 2.5D Temporal dataset is published as S2-cell tiles. Hyderabad falls in the '3d8' prefix region.
# This checks the published tile index responds, without downloading a tile.
IDX = "https://storage.googleapis.com/open-buildings-data/v3/score_thresholds_s2_level_4.csv"
try:
    status, body = get(IDX)
    txt = body.decode("utf-8", "replace").splitlines()
    print("Index reachable. HTTP", status, "-", len(txt), "rows")
    print("First rows:", txt[:3])
    print()
    print("NOTE: reachability is not permission. Before any ingest, re-read the licence page and")
    print("record which of CC BY 4.0 or ODbL we are taking it under, because the obligations differ.")
except Exception as ex:
    print("FAILED -", type(ex).__name__, ex)
    print("Not fatal. Building extrusion is v2; this only tells us whether to plan for it.")

## 7. Verdict

Read this honestly. The question is not "is something available" but "is there enough, at the places
we make claims about, for a reader to check us.

In [ ]:
print("=" * 66)
print("STREET-LEVEL IMAGERY")
if results_mapillary:
    covered = sum(1 for v in results_mapillary.values() if v.get("n", 0) > 0)
    total = len(results_mapillary)
    print(f"  Mapillary: {covered} of {total} probed points have any imagery within {RADIUS_M} m")
    if covered == 0:
        print("  -> No coverage. A street-level layer is NOT available for v1. Do not design for it.")
    elif covered < total / 2:
        print("  -> Patchy. Usable as occasional evidence, never as a promised feature:")
        print("     a 'view this place' button that works a third of the time is worse than none.")
    else:
        print("  -> Workable. Check the years above: imagery older than the roadworks proves nothing.")
else:
    print("  Mapillary: not probed (no token).")

pano_n = sum(v.get("n", 0) for v in results_panoramax.values())
print(f"  Panoramax: {pano_n} images across the sampled points")

print()
print("OSM GROUND FEATURES NEAR THE WORST POINTS")
missing = [k for k, v in results_osm.items() if v == 0]
if missing:
    print(f"  {len(missing)} of {len(results_osm)} points have NO recorded crossing within 400 m:")
    for m in missing:
        print(f"    - {m}")
    print("  -> Either these places genuinely have no crossing, which is the finding,")
    print("     or the map is missing it, which is the correction. Only a visit separates the two.")
else:
    print("  Every probed point has some crossing recorded nearby.")

print()
print("=" * 66)
print("WHAT THIS DOES NOT SETTLE")
print("  Coverage is not verification. An image proves a crossing exists; it cannot prove")
print("  the 2,344 m route is right. Only the driven track does that, and it is still zero.")

In [ ]:
import urllib.request, urllib.parse, json, time
OV = "https://overpass-api.de/api/interpreter"

def q(query):
    d = urllib.parse.urlencode({"data": query}).encode()
    r = urllib.request.Request(OV, data=d, headers={"User-Agent": "arcroad-probe/1.2"})
    return json.loads(urllib.request.urlopen(r, timeout=120).read())

print("CONTROL - places where crossings certainly exist on the ground")
for name, lat, lon in [("Abids / core city", 17.3850, 78.4867),
                       ("Banjara Hills", 17.4126, 78.4350),
                       ("Secunderabad stn", 17.4340, 78.5010),
                       ("Hitec City", 17.4485, 78.3800)]:
    try:
        n = q(f'[out:json][timeout:90];node(around:400,{lat},{lon})["highway"="crossing"];out count;')
        print(f"  {name:22s} {n['elements'][0]['tags']['total']:>5} crossings within 400 m")
    except Exception as e:
        print(f"  {name:22s} FAILED {type(e).__name__}")
    time.sleep(4)

print()
print("CITYWIDE - does OSM record crossings in Hyderabad at all?")
for label, el in [("highway=crossing nodes", 'node(17.05,78.05,17.75,78.85)["highway"="crossing"]'),
                  ("footway ways         ", 'way(17.05,78.05,17.75,78.85)["highway"="footway"]'),
                  ("footbridge ways      ", 'way(17.05,78.05,17.75,78.85)["bridge"]["highway"~"footway|path"]')]:
    try:
        n = q(f'[out:json][timeout:180];{el};out count;')
        print(f"  {label}: {n['elements'][0]['tags']['total']}")
    except Exception as e:
        print(f"  {label}: FAILED {type(e).__name__}")
    time.sleep(4)
